# 01 — ADME EDA (Part 1)

**Goal**: Understand the ADME dataset thoroughly before making any cleaning or modelling decisions.  
**Dataset**: `data/raw/ADME_public_set_3521.csv` — 3521 compounds, 6 log-transformed ADME endpoints.  
**Outputs**: Findings documented in Section 1.10; no data modified here.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.eda import smiles_validity_report, missing_value_report
from src.features import rdkit_descriptors
from src.plotting import endpoint_distributions

SEED = 42
DATA_PATH = '../data/raw/ADME_public_set_3521.csv'

ENDPOINT_COLS = [
    'LOG HLM_CLint (mL/min/kg)',
    'LOG MDR1-MDCK ER (B-A/A-B)',
    'LOG SOLUBILITY PH 6.8 (ug/mL)',
    'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)',
    'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)',
    'LOG RLM_CLint (mL/min/kg)',
]

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK')

## 1.1 — Load & Inspect

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nDtypes:')
print(df.dtypes)
df.head()

## 1.2 — SMILES Validity Report

Report only — no rows dropped here. Dropping is a cleaning decision made after EDA.

In [ ]:
validity = smiles_validity_report(df, smiles_col='SMILES')
print(f"Valid SMILES  : {validity['valid_count']} ({validity['valid_count']/len(df)*100:.2f}%)")
print(f"Invalid SMILES: {validity['invalid_count']} ({validity['invalid_count']/len(df)*100:.2f}%)")
if validity['invalid_indices']:
    print(f"Invalid row indices: {validity['invalid_indices']}")
    print(df.loc[validity['invalid_indices'], ['Internal ID', 'SMILES']])
else:
    print('No invalid SMILES found.')

## 1.3 — Duplicate Check

In [ ]:
from rdkit import Chem

# Duplicate raw SMILES strings
dup_raw = df['SMILES'].duplicated(keep=False)
print(f"Duplicate raw SMILES: {dup_raw.sum()} rows ({df['SMILES'].duplicated().sum()} extra copies)")

# Canonical SMILES duplicates (catches representation variants)
def to_canonical(smi):
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol is not None else None

df['canonical_smiles'] = df['SMILES'].apply(to_canonical)
dup_can = df['canonical_smiles'].duplicated(keep=False) & df['canonical_smiles'].notna()
n_dup_can = df['canonical_smiles'].duplicated().sum()
print(f"Duplicate canonical SMILES: {dup_can.sum()} rows ({n_dup_can} extra copies)")

if n_dup_can > 0:
    dup_df = df[dup_can].sort_values('canonical_smiles')
    # Check endpoint consistency among duplicates
    dup_ep_std = dup_df.groupby('canonical_smiles')[ENDPOINT_COLS].std()
    inconsistent = (dup_ep_std > 0).any(axis=1)
    print(f"Duplicates with inconsistent endpoint values: {inconsistent.sum()}")
    display(dup_df[['Internal ID', 'canonical_smiles'] + ENDPOINT_COLS].head(20))

## 1.4 — Missing Value Report

In [ ]:
miss_report = missing_value_report(df, ENDPOINT_COLS)
print('Missing values per endpoint:')
display(miss_report)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
miss_report['pct_missing'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('% Missing per Endpoint')
ax.set_ylabel('% Missing')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# Heatmap of missingness pattern
miss_matrix = df[ENDPOINT_COLS].isna().astype(int)
short_labels = ['HLM', 'MDR1', 'SOL', 'PPB_H', 'PPB_R', 'RLM']
miss_matrix.columns = short_labels

fig, ax = plt.subplots(figsize=(8, 3))
# Show a row sample — full heatmap on 3521 rows is unreadable
# Instead show missingness co-occurrence as correlation of missingness indicators
miss_corr = miss_matrix.corr()
sns.heatmap(miss_corr, annot=True, fmt='.2f', cmap='Oranges', ax=ax, vmin=0, vmax=1)
ax.set_title('Missingness Co-occurrence (correlation of NaN indicators)')
plt.tight_layout()
plt.show()

## 1.5 — Summary Statistics

In [ ]:
desc = df[ENDPOINT_COLS].describe().T

skew = df[ENDPOINT_COLS].apply(lambda x: stats.skew(x.dropna()))
kurt = df[ENDPOINT_COLS].apply(lambda x: stats.kurtosis(x.dropna()))

desc['skewness'] = skew
desc['kurtosis'] = kurt
desc.index = ['HLM', 'MDR1', 'SOL', 'PPB_H', 'PPB_R', 'RLM']

print('Summary statistics (all 6 endpoints):')
display(desc[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skewness', 'kurtosis']])

## 1.6 — Outlier Detection

Flag compounds >3σ from the mean per endpoint. Report only — no rows removed.

In [ ]:
outlier_summary = {}
outlier_indices = set()

for col in ENDPOINT_COLS:
    col_data = df[col].dropna()
    mu, sigma = col_data.mean(), col_data.std()
    mask = (df[col] - mu).abs() > 3 * sigma
    flagged = df.index[mask & df[col].notna()].tolist()
    outlier_summary[col] = len(flagged)
    outlier_indices.update(flagged)

print('Outlier counts per endpoint (>3σ):')
for k, v in outlier_summary.items():
    print(f"  {k}: {v}")
print(f"\nUnique compounds flagged in any endpoint: {len(outlier_indices)}")

if outlier_indices:
    print('\nFlagged rows (first 20):')
    display(df.loc[sorted(outlier_indices)[:20], ['Internal ID', 'SMILES'] + ENDPOINT_COLS])

## 1.7 — Endpoint Distributions

In [ ]:
fig = endpoint_distributions(df, ENDPOINT_COLS)
plt.show()

**Distribution notes** (fill in after running):
- HLM_CLint:
- MDR1-MDCK ER:
- SOLUBILITY:
- PPB HUMAN:
- PPB RAT:
- RLM_CLint:

## 1.8 — Endpoint Correlations

In [ ]:
corr = df[ENDPOINT_COLS].corr(method='pearson')
corr.index = corr.columns = ['HLM', 'MDR1', 'SOL', 'PPB_H', 'PPB_R', 'RLM']

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, ax=ax, square=True
)
ax.set_title('Pearson Correlation — Endpoints')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter matrix (pairplot)
plot_df = df[ENDPOINT_COLS].copy()
plot_df.columns = ['HLM', 'MDR1', 'SOL', 'PPB_H', 'PPB_R', 'RLM']

g = sns.pairplot(plot_df.dropna(), plot_kws={'alpha': 0.3, 's': 10}, diag_kind='kde')
g.figure.suptitle('Endpoint Scatter Matrix', y=1.02)
plt.show()

## 1.9 — Chemical Space Overview

In [ ]:
# Pre-validate SMILES before featurising — rdkit_descriptors raises on invalid input
assert validity["invalid_count"] == 0, f"Fix {validity['invalid_count']} invalid SMILES before featurising"

# Compute RDKit 2D descriptors (uses only valid SMILES)
desc_df = rdkit_descriptors(df['SMILES'].tolist())
desc_df.index = df.index
print('Descriptor shape:', desc_df.shape)
display(desc_df.describe())

In [ ]:
# Histograms of physicochemical properties
prop_labels = {
    'MW': 'Molecular Weight (Da)',
    'LogP': 'LogP',
    'TPSA': 'TPSA (Å²)',
    'HBD': 'H-Bond Donors',
    'HBA': 'H-Bond Acceptors',
    'RotBonds': 'Rotatable Bonds',
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (col, label) in zip(axes.flatten(), prop_labels.items()):
    data = desc_df[col].dropna()
    ax.hist(data, bins=40, color='teal', edgecolor='white', alpha=0.8)
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
plt.tight_layout()
plt.suptitle('Physicochemical Property Distributions', y=1.02)
plt.show()

In [ ]:
# Lipinski Ro5 violations (informational only)
lip_violations = (
    (desc_df['MW'] > 500).astype(int) +
    (desc_df['LogP'] > 5).astype(int) +
    (desc_df['HBD'] > 5).astype(int) +
    (desc_df['HBA'] > 10).astype(int)
)

print('Lipinski Ro5 violations per compound:')
print(lip_violations.value_counts().sort_index())
print(f"\nCompounds with >=2 violations (Ro5 fails): {(lip_violations >= 2).sum()} "
      f"({(lip_violations >= 2).mean()*100:.1f}%)")

## 1.10 — EDA Conclusions

> **No decisions made here.** Findings are documented; cleaning strategy (SYNC-002) decided post-EDA review.

### Invalid SMILES
- **Finding**: 0 invalid SMILES out of 3521 (100% valid). No action required.

### Duplicates
- **Finding**: 0 duplicate raw SMILES; 0 duplicate canonical SMILES. Dataset is clean on this front.

### Missingness
- **Finding**: Missingness varies dramatically by endpoint:
  - HLM: 434 missing (12.3%), RLM: 467 missing (13.3%) — manageable
  - MDR1: 879 missing (25.0%), SOL: 1348 missing (38.3%) — substantial
  - PPB_HUMAN: 3327 missing (94.5%), PPB_RAT: 3353 missing (95.2%) — near-complete missingness
- **Cleaning strategy to decide (SYNC-002)**: PPB endpoints are nearly empty — likely not useful for modelling. Per-endpoint filtering (train each model on rows with non-missing values for that endpoint) is strongly preferred over complete-cases (which would leave only ~5% of data). Decision after supervisor review.

### Endpoint Distributions & Outliers
- **Skewness**: SOLUBILITY is left-skewed (skew=-1.64); HLM and MDR1 are mildly right-skewed (~0.6–0.8); RLM, PPB endpoints near-symmetric.
- **Outliers (>3σ)**: SOLUBILITY has 20 flagged outliers — largest concern. HLM: 3, MDR1: 1, PPB_RAT: 1. ~25 unique compounds flagged across all endpoints.
- **Endpoints to watch**: SOLUBILITY (high skew + most outliers + high missingness).

### Chemical Space
- **Drug-likeness**: 94.7% of compounds have 0 Lipinski Ro5 violations — dataset is overwhelmingly drug-like.
- **Violations**: 148 compounds have 1 violation, 41 have ≥2 (Ro5 fails). Informational only.

### Next Step
Review findings with supervisor → confirm SYNC-002 cleaning strategy (per-endpoint recommended given PPB missingness) → proceed to Part 2 (baseline models).